In [0]:
from dataclasses import dataclass
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import *
from delta.tables import *

In [0]:
# Create class for Bronze Layer
@dataclass
class BronzeLayer:
    file_path:str
    header:bool
    delimiter:str
    table_name:str

    def __post_init__(self) -> None:
        self.format_type = self.file_path.split('.')[-1]
        self.target_table_bronze = f'{self.table_name}'

    @classmethod
    def from_config_table(cls, pipeline_name:str) -> "BronzeLayer":
        conf = (spark.table("netflix.config_table")
               .filter(col("pipeline_name") == pipeline_name)
               .select("file_path", "header", "delimiter", "table_name")
               .first())
        return cls(
            file_path = conf.file_path
            , header = conf.header
            , delimiter = conf.delimiter
            , table_name = conf.table_name
        )
    def read_from_file(self) -> DataFrame:
        df = (
            spark.read.format(self.format_type)
            .option("header", self.header)
            .option("delimiter", self.delimiter)
            .load(self.file_path)
        )
        # return with another metadata columns
        return (
            df
            .withColumn("_load_dt", current_date())
            .withColumn("_load_dttm", current_timestamp())
            .withColumn("_file_name", col("_metadata.file_name"))
            .withColumn("_file_path", col("_metadata.file_path"))
            .withColumn("_file_size", col("_metadata.file_size"))
            .withColumn("_file_mod", col("_metadata.file_modification_time"))
        )
    def load_to_bronze_table(self, raw_df: DataFrame) -> None:
        # Ensure the bronze table is initialized with correct settings before loading
        self._init_bronze_table() 

        # write data to bronze table
        raw_df.write.mode("append").saveAsTable(self.target_table_bronze)
        print(f"Table {self.target_table_bronze} loaded")

    def _init_bronze_table(self) -> None:
        # created schema for first time and enable CDF to table
        ## 1. First check if table exists or not
        if spark.catalog.tableExists(self.target_table_bronze):
            print(f"Table {self.target_table_bronze} already exists.")
            return # end method immediately
            
        # 2. If table not exitsts. We create table
        else:
            print(f"Table {self.target_table_bronze} does not exist. Initializing...")
            
            # Retrive schema table (read with out data limit 0)
            source_df = self.read_from_file().limit(0)
            
            # Create table from Schema and enable CDF with Python API
            (source_df.write
             .format("delta")
             .option("delta.enableChangeDataFeed", "true")
             .mode("ignore") # prevent other class creating table before this action
             .saveAsTable(self.target_table_bronze))
            
            print(f"Table {self.target_table_bronze} created successfully with CDF enabled.")

In [0]:
# spark.table("workspace.netflix.config_table").display()

In [0]:
# %sql
# truncate table workspace.netflix.netflix_bronze

In [0]:
# Set current schema to netflix to ensure table is created in the correct location
# spark.sql("USE netflix")

# b = BronzeLayer.from_config_table("netflix")
# bronze_df = b.read_from_file()
# b.load_to_bronze_table(bronze_df)

# spark.table("workspace.netflix.netflix_bronze").display()

In [0]:
# %sql
# describe extended  workspace.netflix.netflix_bronze

In [0]:
# spark.table("workspace.netflix.netflix_bronze").display()

In [0]:
# %sql
# DESCRIBE HISTORY workspace.netflix.netflix_bronze

ต้องทำการสำรวจข้อมูลก่อน(ทำแล้ว) โดยเราต้องมีการทำความสะอาดข้อมูล ทำ Normalization Standardization ประมาณนี้
📋 ลำดับการ Clean Data ที่จะใช้ในโปรเจคนี้
    1. Load data from Bronze by use feature like CDF for get only new data(by compare current version and last version of data which processed already in control table) and Add serogete key to table # successes
    2. Trim String Column & Change Data Type
        1.1 Standardize Date & Business 
    3. Get Invalid Record (Data Quality Audit)
    4. Get Key Duplicate (Deduplication Logic)
    5. Get Row Duplicate
    6. Get only bad record and save to bad record table
    7. After get bad record left anti join for get final record
    8. Create hash key and hash value for merge with SCD type 2 with both bad and final Dataframe and Drop column name "_sk"
    9. Need to separate subtable and explode for each subtable then clean and merge we have 4 dimention and 4 bridge table and 1 main dimention table 


In [0]:
# create class for silver layer
@dataclass
class SilverLayer:
    table_name: str
    schema_detail: dict[str, str]
    keys: list[str]
    write_mode: str

    def __post_init__(self) -> None:
        self.bronze_table_name = f"{self.table_name}"
        self.silver_table_name = f"{self.table_name}"
        self.bad_record_table_name = f"{self.table_name}_bad_record"
        self.data_col = [col_name for col_name in self.schema_detail.keys()]
        self.invalid_rule = {"int": "^[0-9]+$", "date": "^\\d{4}-\\d{2}-\\d{2}$"}

    @classmethod
    def from_config_table(cls, pipeline_name: str) -> "SilverLayer":
        conf = (
            spark.table("workspace.netflix.config_table")
            .filter(col("pipeline_name") == pipeline_name)
            .select(
                "table_name", "schema_detail", "keys", "write_mode"
            )
            .first()
        )
        return cls(
            table_name=conf.table_name,
            schema_detail=conf.schema_detail,
            keys=conf.keys,
            write_mode=conf.write_mode,
        )
    # Helper Method for get invalid reason
    def _get_reason(self, df: DataFrame) -> DataFrame:
        control_col = [col_name for col_name in df.columns if col_name.startswith("_") and col_name != "_sk"]
        data_col = [col_name for col_name in df.columns if not col_name.startswith("_")]
        or_statement = " OR ".join([col_name for col_name in control_col])
        return (
            df
            .filter(or_statement)
            .melt(
                ids = [*data_col, "_sk"]
                , values = control_col
                , variableColumnName = "reason"
                , valueColumnName = "status"
            )
        .filter(col("status") == True)
        .groupBy(*data_col, "_sk")
        .agg(collect_list("reason").alias("reason"))
        )
    
    def trim_data(self, df: DataFrame) -> DataFrame:
        """
        Trim all string columns to remove leading/trailing whitespace.
        Uses select() to avoid performance issues from .withColumn() in a loop.
        """
        # Pre-compute schema to avoid repeated Analyze RPCs
        df_columns = df.columns
        
        trim_exprs = [
            trim(col(col_name)).alias(col_name) if col_type == "string" else col(col_name)
            for col_name, col_type in self.schema_detail.items()
        ]
        # Include _sk if it exists
        if "_sk" in df_columns:
            trim_exprs.append(col("_sk"))
        
        return df.select(*trim_exprs)
    
    def change_data_type(self, df: DataFrame) -> DataFrame:
        """
        Change data type of columns based on schema_detail.
        For date columns, uses try_to_date() with "MMMM dd, yyyy" format.
        For other columns, uses try_cast() to return NULL for invalid conversions.
        Records with NULL values can be caught later in get_invalid_record().
        """
        # Pre-compute columns to avoid repeated Analyze RPCs
        df_columns = df.columns
        
        change_type_exprs = []
        
        for col_name, col_type in self.schema_detail.items():
            # Generic date handling - assumes "MMMM dd, yyyy" format for all date columns
            if col_type == "date":
                change_type_exprs.append(
                    expr(f"try_to_date({col_name}, 'MMMM dd, yyyy')").alias(col_name)
                )
            else:
                change_type_exprs.append(
                    expr(f"try_cast({col_name} as {col_type})").alias(col_name)
                )
        
        # Include _sk if it exists (using pre-computed df_columns)
        if "_sk" in df_columns:
            change_type_exprs.append(col("_sk"))
        
        return df.select(*change_type_exprs)
    
    def get_invalid_record(self, bronze_df: DataFrame) -> DataFrame:
        invalid_col = {
            f"_is_{col_name}_invalid": coalesce(~col(col_name).rlike(self.invalid_rule[col_type]), lit(False))
            for col_name, col_type in self.schema_detail.items() if col_type in ["int", "date"]
        }
        
        return (
            bronze_df
            .withColumns(invalid_col)
            .transform(self._get_reason)
        )
    
    def get_key_null_record(self, bronze_df: DataFrame) -> DataFrame:
        key_null_statement = { f"_is_{col_name}_null": col(col_name).isNull() for col_name in self.keys}

        return (
            bronze_df.withColumns(key_null_statement)
            .transform(self._get_reason)
        )
    
    def get_dup_record(self, bronze_df: DataFrame, key_null_df: DataFrame) -> DataFrame:
        partition_by_all = Window.partitionBy(*self.data_col).orderBy("_sk")
        partition_by_key = Window.partitionBy(*self.keys)

        bronze_not_null_df = bronze_df.join(key_null_df, ['_sk'], "left_anti")

        is_row_duplicate_df = (
            bronze_not_null_df
            .withColumn("rn", row_number().over(partition_by_all))
            .filter(col("rn") > 1)
            .drop("rn")
            .withColumn("reason", array(lit("_row_duplication")))
        )

        is_key_duplication_df = (
            bronze_not_null_df
            .join(is_row_duplicate_df, ['_sk'], "left_anti")
            .withColumn("count", count("*").over(partition_by_key))
            .filter(col("count") > 1)
            .drop("count")
            .withColumn("reason", array(lit("_key_duplicate")))
        )
        return (
            is_row_duplicate_df
            .unionByName(is_key_duplication_df)
        )

    def get_all_bad_record(self, invalid_df: DataFrame, key_null_df: DataFrame, duplicate_df: DataFrame) -> DataFrame:
        return (
            invalid_df
            .unionByName(key_null_df)
            .unionByName(duplicate_df)
            .groupBy(*self.data_col, "_sk")
            .agg(flatten(collect_list("reason")).alias("reason"))
        )

    def get_final_result(self, bronze_df: DataFrame, all_bad_df: DataFrame) -> DataFrame:
        add_control_col = {"load_dt" : current_date()
                           , "load_dttm": current_timestamp()}
        return (
            bronze_df
            .join(all_bad_df, ["_sk"], "left_anti")
            .select(*self.data_col, "_sk")
            .withColumns(add_control_col)
        )

    def get_hash_key_value(self, final_df: DataFrame) -> DataFrame:
        # These column use to explode so we don't use them in the hash key
        columns_to_explode = ["cast", "director", "country", "listed_in"]
        columns_to_hash = [col_name for col_name in self.data_col 
                           if col_name not in self.keys and col_name not in columns_to_explode]
        # Create hash key and hash value
        df_with_hash = (
            final_df
            .withColumn("hash_key", sha2(concat_ws("||", *[col(key) for key in self.keys]), 256))
            .withColumn("hash_value", sha2(concat_ws("||", *[col(value) for value in columns_to_hash]), 256))
        )
        columns_to_drop = columns_to_explode + ["_sk"]
        # Drop columns we don't need
        final_main_dimension_df = df_with_hash.drop(*columns_to_drop)
        return final_main_dimension_df


    def load_bad_record(self, all_bad_df: DataFrame, batch_id: int) -> None:
        """
        Load bad records to the bad record table with metadata.
        Appends records with batch_id, load_dt, and load_dttm for tracking.
        """
        if all_bad_df.isEmpty():
            print(f"Batch {batch_id}: No bad records to load")
            return
        
        # Add metadata columns for tracking
        bad_records_with_metadata = (
            all_bad_df
            .withColumn("batch_id", lit(batch_id))
            .withColumn("load_dt", current_date())
            .withColumn("load_dttm", current_timestamp())
        )
        
        # Write to bad record table
        bad_records_with_metadata.write.mode("append").saveAsTable(self.bad_record_table_name)
        
        print(f"Batch {batch_id}: Loaded {all_bad_df.count()} bad records to {self.bad_record_table_name}")

    def load_to_silver_layer(self, final_result_df: DataFrame, batch_id: int) -> None:
        """
        Load final results to silver table.
        batch_id: The batch number from foreachBatch (for logging/debugging).
        """
        pass # I will create for next day 18/06/2026


    def process_cdf_stream_to_silver(self, checkpoint_location: str = None) -> None:
        """
        Process CDF stream from bronze to silver with quality checks.
        Uses trigger(availableNow=True) for serverless-friendly incremental processing.
        """
        # Use provided checkpoint location or default to table-specific path
        if checkpoint_location is None:
            checkpoint_location = f"/Volumes/workspace/netflix/checkpoint_dir/{self.table_name}_silver/"
            
        # Create a stream from the bronze table
        # Note: _sk is added in _process_quality_checks_batch to avoid collision across batches
        cdf_stream = (
            spark.readStream
            .option("readChangeFeed", "true")
            .option("startingVersion", 0)  # Start from version 0 or use checkpoint
            .table(self.bronze_table_name)
            .filter(col("_change_type").isin(["insert", "update_postimage"]))
            .select(*self.data_col)
        )

        query = (
            cdf_stream.writeStream
            .foreachBatch(self._process_quality_checks_batch)
            .option("checkpointLocation", checkpoint_location)
            .outputMode("append") # Only use when we use foreachBatch 
            .trigger(availableNow=True)
            .start()
        )
    
        query.awaitTermination()
        print(f"Stream processing complete for {self.silver_table_name}")

    def _process_quality_checks_batch(self, batch_df: DataFrame, batch_id: int) -> None:
        """
        Process each micro-batch through quality checks.
        Called automatically by foreachBatch with (batch_df, batch_id).
        """
        if batch_df.isEmpty():
            return
        
        # Add unique surrogate key: combine batch_id with monotonically_increasing_id()
        # This ensures _sk is unique across all batches
        batch_with_sk = batch_df.withColumn(
            "_sk",
            (lit(batch_id).cast("long") * 1000000000) + monotonically_increasing_id()
        )
        
        # Standardize and clean DataFrame
        trimmed_stream = self.trim_data(batch_with_sk)
        change_data_type_stream = self.change_data_type(trimmed_stream)
        invalid_df = self.get_invalid_record(change_data_type_stream)
        key_null_df = self.get_key_null_record(change_data_type_stream)
        duplicate_df = self.get_dup_record(change_data_type_stream, key_null_df)
        all_bad_df = self.get_all_bad_record(invalid_df, key_null_df, duplicate_df)
        final_df = self.get_final_result(change_data_type_stream, all_bad_df)

        # Load DataFrame into 2 tables: Silver table and Bad_record table
        self.load_bad_record(all_bad_df, batch_id)
        self.load_to_silver_layer(final_df, batch_id)
        
        print(f"Batch {batch_id}: Complete")


In [0]:
# spark.table("workspace.netflix.netflix_bronze").display()

In [0]:
# spark.table("workspace.netflix.config_table").display()

In [0]:
# Test all methods in one cell
print("=" * 50)
print("TESTING SILVER LAYER METHODS")
print("=" * 50)

# 1. Initialize SilverLayer
print("\n1. Initialize SilverLayer")
silver = SilverLayer.from_config_table("netflix")
print(f"   Table: {silver.table_name}")
print(f"   Keys: {silver.keys}")

# 2. Load test data
print("\n2. Load test data from bronze")
test_df = (
    spark.table(silver.bronze_table_name)
    .limit(10)
    .select(*silver.data_col)
    .withColumn("_sk", monotonically_increasing_id())
)
print(f"   Loaded: {test_df.count()} records")
print("\nOriginal Data:")
display(test_df)

# 3. Test trim_data
print("\n3. Test trim_data method")
trimmed_df = silver.trim_data(test_df)
print("   ✅ trim_data completed")

# 4. Test change_data_type
print("\n4. Test change_data_type method")
typed_df = silver.change_data_type(trimmed_df)
print(f"   Schema after type conversion: {typed_df.dtypes}")
print("\nAfter Type Conversion:")
display(typed_df)

# 5. Test get_invalid_record
print("\n5. Test get_invalid_record method")
invalid_df = silver.get_invalid_record(typed_df)
print(f"   Invalid records found: {invalid_df.count()}")
if invalid_df.count() > 0:
    print("\nInvalid Records:")
    display(invalid_df)
else:
    print("   ✅ No invalid records found")

# 6. Test get_key_null_record
print("\n6. Test get_key_null_record method")
key_null_df = silver.get_key_null_record(typed_df)
print(f"   Key null records found: {key_null_df.count()}")
if key_null_df.count() > 0:
    print("\nKey Null Records:")
    display(key_null_df)
else:
    print("   ✅ No null key records found")

# 7. Test get_dup_record
print("\n7. Test get_dup_record method")
try:
    duplicate_df = silver.get_dup_record(typed_df, key_null_df)
    print("   Duplicate check logic executed successfully")
    dup_count = duplicate_df.count()
    print(f"   Duplicate records found: {dup_count}")
    if dup_count > 0:
        print("\nDuplicate Records:")
        display(duplicate_df)
    else:
        print("   ✅ No duplicate records found")
except Exception as e:
    print(f"   ⚠️  Error in get_dup_record: {str(e)}")
    # Create empty DataFrame with same schema as invalid_df/key_null_df (data_col + _sk + reason)
    from pyspark.sql.types import ArrayType, StringType
    duplicate_df = spark.createDataFrame([], schema=invalid_df.schema)
    print("   Using empty duplicate_df for testing get_all_bad_record")

# 8. Test get_all_bad_record
print("\n8. Test get_all_bad_record method")
all_bad_df = silver.get_all_bad_record(invalid_df, key_null_df, duplicate_df)
print(f"   Total bad records: {all_bad_df.count()}")
if all_bad_df.count() > 0:
    print("\nAll Bad Records (combined):")
    display(all_bad_df)
    print("\nBad Records Summary by Reason:")
    all_bad_df.withColumn("reason_str", concat_ws(", ", col("reason"))).groupBy("reason_str").count().display()
else:
    print("   ✅ No bad records found")

# 9. Test get_final_result
print("\n9. Test get_final_result method")
final_result_df = silver.get_final_result(typed_df, all_bad_df)
print(f"   Final result records: {final_result_df.count()}")
print(f"   Expected records: {typed_df.count()} (original) - {all_bad_df.count()} (bad) = {typed_df.count() - all_bad_df.count()}")

if final_result_df.count() > 0:
    print("\n✅ Final Result DataFrame:")
    display(final_result_df)
    print("\nFinal Result Schema:")
    final_result_df.printSchema()
    print("\n✅ Control columns added successfully (load_dt, load_dttm)")
else:
    print("   ⚠️  Warning: No records in final result")

# 10. Test get_hash_key_value
print("\n10. Test get_hash_key_value method")
try:
    hashed_df = silver.get_hash_key_value(final_result_df)
    print("   ✅ Hash key and value created successfully")
    print(f"   Records after hashing: {hashed_df.count()}")
    
    # Check that hash columns were added
    hash_columns = [col for col in hashed_df.columns if 'hash' in col]
    print(f"   Hash columns added: {hash_columns}")
    
    # Check that columns were dropped
    columns_to_explode = ["cast", "director", "country", "listed_in"]
    dropped_cols = [col for col in columns_to_explode if col not in hashed_df.columns]
    print(f"   Columns dropped: {dropped_cols + ['_sk'] if '_sk' not in hashed_df.columns else dropped_cols}")
    
    print("\n✅ Final Main Dimension DataFrame (with hash):")
    display(hashed_df)
    print("\nFinal Main Dimension Schema:")
    hashed_df.printSchema()
    
except Exception as e:
    print(f"   ⚠️  Error in get_hash_key_value: {str(e)}")
    import traceback
    traceback.print_exc()

# 11. Test load_bad_record
print("\n11. Test load_bad_record method")
test_batch_id = 999  # Use a test batch ID

# First, check if bad record table exists and show count before
try:
    before_count = spark.table(silver.bad_record_table_name).count()
    print(f"   Records in {silver.bad_record_table_name} before: {before_count}")
except:
    before_count = 0
    print(f"   Table {silver.bad_record_table_name} does not exist yet")

# Test with the all_bad_df from step 8
if all_bad_df.count() > 0:
    print(f"   Loading {all_bad_df.count()} bad records with batch_id={test_batch_id}")
    silver.load_bad_record(all_bad_df, test_batch_id)
    
    # Verify the load
    after_count = spark.table(silver.bad_record_table_name).count()
    print(f"   Records in {silver.bad_record_table_name} after: {after_count}")
    print(f"   Records added: {after_count - before_count}")
    
    print("\n✅ Bad Record Table Sample (latest batch):")
    spark.table(silver.bad_record_table_name).filter(col("batch_id") == test_batch_id).display()
    
    print("\nBad Record Table Schema:")
    spark.table(silver.bad_record_table_name).printSchema()
else:
    print("   ✅ No bad records to test - method would return early")
    print("   Testing with empty DataFrame...")
    silver.load_bad_record(all_bad_df, test_batch_id)

print("\n" + "=" * 50)
print("ALL TESTS COMPLETED SUCCESSFULLY! ✅")
print("=" * 50)

In [0]:
# Test load_bad_record with intentionally bad data
print("="*70)
print("TESTING LOAD_BAD_RECORD WITH BAD DATA")
print("="*70)

# Initialize SilverLayer
silver = SilverLayer.from_config_table("netflix")

# Create test data with various types of bad records
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("release_year", StringType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True),
])

bad_data = [
    # 1. Invalid date format (should be "MMMM dd, yyyy" but is "yyyy-MM-dd")
    ("bad1", "Movie", "Bad Date Movie", "Director A", "Actor A", "USA", 
     "2021-09-25", "2020", "PG-13", "90 min", "Drama", "Test movie with invalid date format"),
    
    # 2. Invalid integer (release_year contains letters)
    ("bad2", "TV Show", "Bad Year Show", "Director B", "Actor B", "UK", 
     "September 25, 2021", "202X", "TV-MA", "2 Seasons", "Comedy", "Test show with invalid year"),
    
    # 3. Null key (show_id is null)
    (None, "Movie", "Null Key Movie", "Director C", "Actor C", "Canada", 
     "September 25, 2021", "2019", "R", "120 min", "Action", "Test movie with null key"),
    
    # 4. Row duplicate (exact same as #4)
    ("dup1", "Movie", "Duplicate Movie", "Director D", "Actor D", "France", 
     "September 25, 2021", "2018", "PG", "95 min", "Adventure", "Test duplicate movie"),
    ("dup1", "Movie", "Duplicate Movie", "Director D", "Actor D", "France", 
     "September 25, 2021", "2018", "PG", "95 min", "Adventure", "Test duplicate movie"),
    
    # 5. Key duplicate (same show_id, different content)
    ("dup2", "Movie", "First Version", "Director E", "Actor E", "Germany", 
     "September 25, 2021", "2017", "PG-13", "100 min", "Thriller", "First version of movie"),
    ("dup2", "Movie", "Second Version", "Director F", "Actor F", "Spain", 
     "September 26, 2021", "2017", "PG-13", "105 min", "Thriller", "Second version of movie"),
    
    # 6. Multiple issues (null key + invalid date)
    (None, "TV Show", "Multiple Issues", "Director G", "Actor G", "Italy", 
     "2021-09-25", "2016", "TV-14", "3 Seasons", "Documentary", "Test with multiple quality issues"),
    
    # 7-8. Good records for comparison
    ("good1", "Movie", "Good Movie", "Director H", "Actor H", "Japan", 
     "September 25, 2021", "2015", "PG", "110 min", "Animation", "Test good movie"),
    ("good2", "TV Show", "Good Show", "Director I", "Actor I", "Korea", 
     "September 25, 2021", "2014", "TV-PG", "1 Season", "Reality", "Test good show"),
]

test_bad_df = spark.createDataFrame(bad_data, schema=schema)
print(f"\n✅ Created test dataset with {test_bad_df.count()} records")
print("   - 2 invalid data type records (bad1, bad2)")
print("   - 2 null key records (show_id = null)")
print("   - 2 exact duplicates (dup1)")
print("   - 2 key duplicates (dup2)")
print("   - 2 good records (good1, good2)")

print("\n📋 Test Data:")
test_bad_df.display()

# Add surrogate key
test_batch_id = 888
test_with_sk = test_bad_df.withColumn(
    "_sk",
    (lit(test_batch_id).cast("long") * 1000000000) + monotonically_increasing_id()
)

print("\n" + "="*70)
print("RUNNING QUALITY CHECKS")
print("="*70)

# Run through quality checks
trimmed = silver.trim_data(test_with_sk)
typed = silver.change_data_type(trimmed)

print("\n1️⃣ Testing get_invalid_record...")
invalid = silver.get_invalid_record(typed)
print(f"   Invalid records found: {invalid.count()}")
if invalid.count() > 0:
    print("\n   Invalid Records Detail:")
    invalid.select("show_id", "title", "date_added", "release_year", "reason").display()

print("\n2️⃣ Testing get_key_null_record...")
key_null = silver.get_key_null_record(typed)
print(f"   Null key records found: {key_null.count()}")
if key_null.count() > 0:
    print("\n   Null Key Records Detail:")
    key_null.select("show_id", "title", "reason").display()

print("\n3️⃣ Testing get_dup_record...")
duplicate = silver.get_dup_record(typed, key_null)
print(f"   Duplicate records found: {duplicate.count()}")
if duplicate.count() > 0:
    print("\n   Duplicate Records Detail:")
    duplicate.select("show_id", "title", "reason", "_sk").display()

print("\n4️⃣ Testing get_all_bad_record...")
all_bad = silver.get_all_bad_record(invalid, key_null, duplicate)
print(f"   Total bad records: {all_bad.count()}")
if all_bad.count() > 0:
    print("\n   All Bad Records (Consolidated):")
    all_bad.select("show_id", "title", "reason").display()
    
    print("\n   Bad Records Summary by Reason:")
    all_bad.withColumn("reason_str", concat_ws(", ", col("reason"))).groupBy("reason_str").count().orderBy(desc("count")).display()

print("\n5️⃣ Testing get_final_result...")
final = silver.get_final_result(typed, all_bad)
print(f"   Final good records: {final.count()}")
print(f"   Expected: {typed.count()} - {all_bad.count()} = {typed.count() - all_bad.count()}")
if final.count() > 0:
    print("\n   Final Good Records:")
    final.select("show_id", "title", "date_added", "release_year", "load_dt", "load_dttm").display()

print("\n" + "="*70)
print("TESTING LOAD_BAD_RECORD METHOD")
print("="*70)

# Check bad record table before
try:
    before_count = spark.table(silver.bad_record_table_name).count()
    print(f"\n📊 Records in {silver.bad_record_table_name} before: {before_count}")
except:
    before_count = 0
    print(f"\n📊 Table {silver.bad_record_table_name} does not exist yet (will be created)")

# Load bad records
print(f"\n💾 Loading {all_bad.count()} bad records with batch_id={test_batch_id}...")
silver.load_bad_record(all_bad, test_batch_id)

# Verify the load
after_count = spark.table(silver.bad_record_table_name).count()
print(f"\n✅ Records in {silver.bad_record_table_name} after: {after_count}")
print(f"✅ Records added: {after_count - before_count}")

print(f"\n📋 Bad Record Table (batch_id={test_batch_id}):")
spark.table(silver.bad_record_table_name).filter(col("batch_id") == test_batch_id).select(
    "show_id", "title", "reason", "batch_id", "load_dt", "load_dttm"
).display()

print("\n📐 Bad Record Table Schema:")
spark.table(silver.bad_record_table_name).printSchema()

print("\n" + "="*70)
print("TEST COMPLETED SUCCESSFULLY! ✅")
print("="*70)